# ⚡ GridMind-RL: Training an LLM Energy Controller with Unsloth + GRPO

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** to manage industrial building energy
using Reinforcement Learning via the live **GridMind-RL OpenEnv** environment.

| | |
|---|---|
| **Environment** | https://lo-kyu-gridmind.hf.space |
| **Method** | GRPO (Group Relative Policy Optimization) |
| **Framework** | Unsloth (4-bit LoRA) + HF TRL |
| **Model** | unsloth/Qwen2.5-1.5B-Instruct |

### What does the agent learn?
- **Task 1**: Minimize energy cost by charging thermal storage off-peak
- **Task 2**: Maintain indoor temperature while minimizing cost
- **Task 3**: Full demand-response — cost + temperature + grid stress + batch scheduling + carbon

In [ ]:
%%capture
!pip install unsloth openenv-core
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton
!pip install --no-deps cut_cross_entropy unsloth_zoo
!pip install "datasets>=3.4.1,<4.0.0" pandas matplotlib nest_asyncio

## Step 1 — Verify the Live Environment

In [ ]:
from openenv.core import GenericEnvClient
import asyncio, nest_asyncio
nest_asyncio.apply()

ENV_URL = "https://lo-kyu-gridmind.hf.space"

async def verify_env():
    async with GenericEnvClient(base_url=ENV_URL) as env:
        r = await env.reset()
        print("✅ Environment live!")
        print("Observation keys:", list(r.observation.keys()))
        r2 = await env.step({
            "hvac_power_level": 0.5, "thermal_charge_rate": 0.0,
            "batch_job_slot": 0, "load_shed_fraction": 0.0, "building_id": 0
        })
        print(f"Step reward: {r2.reward:.3f}, done: {r2.done}")

asyncio.run(verify_env())

## Step 2 — Load Model with Unsloth 4-bit LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
lora_rank = 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✅ Model loaded with Unsloth 4-bit LoRA")

## Step 3 — Define Reward Functions

We use a **composite reward** with three components:

| Reward Function | Max Score | What it checks |
|---|---|---|
| `reward_valid_json` | 0.3 | Model outputs parsable JSON |
| `reward_has_required_keys` | 0.3 | JSON contains all 4 action fields |
| `reward_env_interaction` | 0.4 | Live environment step reward |
| **Total** | **1.0** | |

In [ ]:
import json, re

def reward_valid_json(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        try:
            match = re.search(r'\{.*?\}', text, re.DOTALL)
            if match:
                json.loads(match.group())
                rewards.append(0.3)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

def reward_has_required_keys(completions, **kwargs):
    required = {"hvac_power_level", "thermal_charge_rate", "batch_job_slot", "load_shed_fraction"}
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        try:
            match = re.search(r'\{.*?\}', text, re.DOTALL)
            if match:
                action = json.loads(match.group())
                rewards.append(0.3 if required.issubset(action.keys()) else 0.1)
            else:
                rewards.append(0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

def reward_env_interaction(completions, **kwargs):
    """Reward 0.0-0.4 based on actual environment reward from live GridMind-RL HF Space."""
    async def run_step(text):
        try:
            match = re.search(r'\{.*?\}', text, re.DOTALL)
            action = json.loads(match.group()) if match else {}
            step_action = {
                "hvac_power_level":    float(max(0, min(1, action.get("hvac_power_level", 0.5)))),
                "thermal_charge_rate": float(max(-1, min(1, action.get("thermal_charge_rate", 0.0)))),
                "batch_job_slot":      int(max(0, min(4, action.get("batch_job_slot", 0)))),
                "load_shed_fraction":  float(max(0, min(0.5, action.get("load_shed_fraction", 0.0)))),
                "building_id": 0
            }
            async with GenericEnvClient(base_url=ENV_URL) as env:
                await env.reset()
                result = await env.step(step_action)
                # Normalize raw env reward (~[-2, 3]) → (0.0, 0.4)
                return min(0.4, max(0.0, (result.reward + 2.0) * 0.08))
        except Exception:
            return 0.0

    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        rewards.append(asyncio.run(run_step(text)))
    return rewards

print("✅ Reward functions defined")
print("  Total max reward per step: 1.0")

## Step 4 — Build Training Dataset & Start GRPO Training

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import pandas as pd, os
from transformers import TrainerCallback

SYSTEM_PROMPT = """You are an expert industrial building energy controller.
Each turn you receive the current building state and must respond with 
ONLY a valid JSON action object.

Action format:
{"hvac_power_level": <0.0-1.0>, "thermal_charge_rate": <-1.0 to 1.0>, 
 "batch_job_slot": <0-4>, "load_shed_fraction": <0.0-0.5>, "building_id": 0}

Strategy:
- Charge storage when price < $0.08/kWh (positive thermal_charge_rate)
- Discharge storage when price > $0.15/kWh (negative thermal_charge_rate)  
- Shed load 0.3-0.5 when grid_stress_signal > 0.7
- Reduce HVAC during peak hours (8-12, 17-21)
- Keep temperature between 19-23°C"""

def make_prompt(i):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",
             "content": f"Episode {i+1}: Building simulation starting. Output your first action as JSON."}]

dataset = Dataset.from_dict({"prompt": [make_prompt(i) for i in range(300)]})
print(f"✅ Dataset: {len(dataset)} training prompts")

# --- CSV Logger ---
log_history = []
class CSVLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            entry = {**logs, "step": state.global_step}
            log_history.append(entry)
            os.makedirs("results", exist_ok=True)
            pd.DataFrame(log_history).to_csv("results/training_log.csv", index=False)

training_args = GRPOConfig(
    output_dir="gridmind-grpo-unsloth",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=256,
    max_completion_length=128,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=100,
    fp16=True,
    report_to="none",
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[reward_valid_json, reward_has_required_keys, reward_env_interaction],
    callbacks=[CSVLogger()]
)

print("🚀 Starting GRPO training...")
trainer.train()

## Step 5 — Plot Training Curve

This plot is the key **evidence of learning** for the hackathon judges.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("results/training_log.csv")
reward_cols = [c for c in df.columns if c.startswith("reward")]

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#1A535C']
for idx, col in enumerate(reward_cols):
    smoothed = df[col].rolling(window=3, min_periods=1).mean()
    label = col.replace('reward_', '').replace('_', ' ').title()
    ax.plot(df['step'], smoothed, label=label, linewidth=2.5, color=colors[idx % len(colors)])

ax.set_title("GridMind-RL Training Curve (Unsloth GRPO)", fontsize=15, pad=15)
ax.set_xlabel("Training Steps")
ax.set_ylabel("Reward Score")
ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig("results/training_curve.png", dpi=200, bbox_inches='tight')
plt.show()
print("✅ Training curve saved to results/training_curve.png")

## Step 6 — Before vs After Comparison

Test the same scenario pre-training and post-training to show qualitative improvement.

In [ ]:
test_state = (
    "Building state: temp=24.5°C (too hot!), price=$0.18/kWh (peak), "
    "storage=0.7 (charged), grid_stress=0.85 (CRITICAL!), hour=18, step=60/95\n"
    "Pending batch job deadlines: [12, 30]\n"
    "Cumulative cost so far: $1.24"
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_state}
]

FastLanguageModel.for_inference(model)
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        inputs, max_new_tokens=100, temperature=0.1,
        do_sample=True, pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("📋 Test Scenario:")
print(" ", test_state.replace("\n", "\n  "))
print("\n🤖 Fine-tuned Model Response:")
print(" ", response)
print("\n✅ Expected: load_shed_fraction > 0 (grid_stress=0.85), thermal_charge_rate < 0 (discharge at peak price)")